# State Management with StateMachine
This notebook demonstrates how to work with state management using a StateMachine implementation. We'll explore how to create, manage, and control workflow states
in a structured way.

## What we'll learn:
- Basic state machine concepts and implementation
- Creating and connecting workflow steps
- Managing state transitions and data flow
- Working with routing and loops in state machines
- Understanding state snapshots and execution flow

### SetUp


In [1]:
from typing import TypedDict
from lib.state_machine import (
StateMachine,
Step,
EntryPoint,
Termination,
)

## Basic State Machine Concepts
Let's start with a simple example that demonstrates the core concepts of our state machine:
1. Defining state schema
2. Creating steps
3. Connecting steps
4. Running the workflow

## Creating the Schema and the state machine

In [2]:
class Schema(TypedDict):
    """Schema defining the structure of our state.

    Attributes:
    input: The input value to process
    output: The processed output value

    input: int
    output: int
    """
    input: int
    output: int

In [3]:
# Create our state machine instance
workflow = StateMachine(Schema)

** Defining the logic for Steps **

In [ ]:
def step_input(state: Schema) -> Schema:
    """First step: Increment the input value.

    Args:
    state: Current state containing input value

    Returns:
        Updated state with incremented value in output
    """
    # random = 10 will be thrown, I have no idea why someone added it in the first place
    # , but it is not part of the Schema and will not cause a runtime error.
    return {"output": state["input"] + 1, "random": 10}

In [6]:
def step_double(state: Schema) -> Schema:
    """Second step: Double the previous output.

    Args :
    state: Current state containing output from previous step

    Returns :
        Updated state with doubled output value
    """
    return {"output": state["output"] * 2}



**Creating and Connecting Steps**

In [ ]:
# This is just a marker step, It automatically sets its step_id to "entry" and uses a dummy function lambda x: {} 
# (it doesn't change any data; it just acts as the "Start" flag).
entry = EntryPoint()
# Define the two steps in our workflow
s1 = Step("input", step_input)
s2 = Step("double", step_double)
# This is also a marker step, It automatically sets its step_id to "termination" and uses a dummy function lambda x: {} 
# (it doesn't change any data; it just acts as the "End" flag).
termination = Termination()

In [25]:
workflow.add_steps([entry, s1, s2, termination])

In [26]:
workflow. connect(entry, s1)
workflow.connect(s1, s2)
workflow.connect(s2, termination)

In [27]:
workflow.transitions

{'entry': [Transition('entry' -> ['increment']),
  Transition('entry' -> ['input'])],
 'increment': [Transition('increment' -> ['increment', 'termination'])],
 'input': [Transition('input' -> ['double'])],
 'double': [Transition('double' -> ['termination'])]}

**Running the Workflow**

In [11]:
initial_state = {"input": 4}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: entry
[StateMachine] Executing step: input
[StateMachine] Executing step: double
[StateMachine] Terminating: termination


Run('164c2033-5b86-44f5-b28e-221e710df0b3')

In [12]:
run_object.snapshots

[Snapshot('62d77d02-3bbb-4d7a-aaac-00895571b1e9') @ [2026-03-05 20:44:40.891832]: entry.State({'input': 4}),
 Snapshot('5a0797d7-67f1-4923-97e8-bc78dbe3e8f2') @ [2026-03-05 20:44:40.891891]: input.State({'input': 4, 'output': 5}),
 Snapshot('90eeed0f-7dd6-4b86-af29-ac73f82afb9f') @ [2026-03-05 20:44:40.891931]: double.State({'input': 4, 'output': 10})]

## Advanced State Management: Routing and Loops
Now we'll explore more complex state management patterns including:
- Conditional routing between steps
- Creating loops in the workflow
- Managing state through multiple iterations

In [13]:
class CounterSchema(TypedDict):
    """Schema for a counter-based workflow.

    Attributes:
        count: Current counter value
        max_value: Maximum value before termination
    """
    count: int
    max_value: int

In [14]:
workflow = StateMachine(CounterSchema)

In [28]:
def increment_counter(state: CounterSchema) -> CounterSchema:
    """Increment the counter value.

    Args:
    state: Current state with counter value

    Returns:
        Updated state with incremented counter
    """

    return {"count": state["count"] + 1}

In [29]:
# Create steps
entry = EntryPoint()
increment = Step("increment", increment_counter)
termination = Termination()

In [30]:
workflow.add_steps([entry, increment, termination])

In [19]:
# Router logic
def check_counter(state: CounterSchema) -> Step:
    """Determine next step based on counter value.

    Args:
    state: Current state with counter and max value

    Returns :
    Next step to execute (increment or terminate)
    """

    if state["count"] >= state["max_value"]:
        return termination
    return increment

In [20]:
# Connect steps with a loop in increment
workflow.connect(entry, increment)
workflow.connect(increment, [increment, termination], check_counter)

In [31]:
workflow.transitions

{'entry': [Transition('entry' -> ['increment']),
  Transition('entry' -> ['input'])],
 'increment': [Transition('increment' -> ['increment', 'termination'])],
 'input': [Transition('input' -> ['double'])],
 'double': [Transition('double' -> ['termination'])]}

In [22]:
initial_state = {"count": 0, "max_value": 3}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: entry
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Terminating: termination


Run('ec2c9ee9-5c8c-44ee-b3e8-a19e6c34474b')

In [23]:
run_object.snapshots

[Snapshot('2fefd056-d712-433d-b77e-6db43a6a8421') @ [2026-03-05 20:45:04.449443]: entry.State({'count': 0, 'max_value': 3}),
 Snapshot('34882898-3f7e-491e-a1bc-7cdc01c8f4ba') @ [2026-03-05 20:45:04.449508]: increment.State({'count': 1, 'max_value': 3}),
 Snapshot('b7157dce-0b48-4f92-a4b4-d4a99390e516') @ [2026-03-05 20:45:04.449547]: increment.State({'count': 2, 'max_value': 3}),
 Snapshot('53cfbd08-8670-4c5d-9275-e464cb15c751') @ [2026-03-05 20:45:04.449578]: increment.State({'count': 3, 'max_value': 3})]

In [24]:
run_object.get_final_state()

{'count': 3, 'max_value': 3}